In [ ]:
#| default_exp game/web

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import httpx
import random

In [ ]:
#| export
import logging

for logger_name in ("uvicorn", "uvicorn.error", "uvicorn.access"):
    uv_logger = logging.getLogger(logger_name)
    file_handler = logging.FileHandler('HexServer.txt')
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    uv_logger.addHandler(file_handler)


In [ ]:
#| export
daisy_hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css')
)

In [ ]:
#| export
app = FastHTML(hdrs=daisy_hdrs)
rt = app.route

In [ ]:
#| export
def get_preview(app):
    return partial(HTMX, app=app, host=None, port=None)
preview = get_preview(app)

In [ ]:
srv = JupyUvi(app=app)

In [ ]:
#| export
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion , windy_edge , unique_windy_edge
from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain
Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins, Watershed

In [ ]:
#| export
from HexMagic.game.kingdom import GameBoard,Kingdom,TradeRoute,Terrain,StyleCSS, Hex, TerraDemo, Geology, DrainageBasins, CountryFlag
from HexMagic.game.country import CountryDetails
from HexMagic.game.store import HexServer

In [ ]:
#| export
import logging

logging.basicConfig(
    filename='base.text',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logging.info("getting Started")


In [ ]:
!tail base.text

Is there a way to get srv to log errors to a file?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! JupyUvi uses uvicorn under the hood. You can configure uvicorn's logging by passing parameters when creating the server. Here's one approach:

```python
srv = JupyUvi(app=app, log_level="debug")
```

However, to get errors written to your file, you'd want to configure uvicorn's loggers to use your file handler. Try adding this before creating `srv`:

```python
import logging

# Get uvicorn's loggers
for logger_name in ("uvicorn", "uvicorn.error", "uvicorn.access"):
    uv_logger = logging.getLogger(logger_name)
    uv_logger.handlers = []  # Clear default handlers
    file_handler = logging.FileHandler('base.text')
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    uv_logger.addHandler(file_handler)
```

Would you like to try that, or would you prefer a different approach (like a custom exception handler in your FastHTML app)?

so I am ok if it is do a different file. Can it log errors to "HexServer.txt" how would I iniate that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! Add this before creating `srv`:

```python
import logging

for logger_name in ("uvicorn", "uvicorn.error", "uvicorn.access"):
    uv_logger = logging.getLogger(logger_name)
    file_handler = logging.FileHandler('HexServer.txt')
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    uv_logger.addHandler(file_handler)
```

Then create your server as usual with `srv = JupyUvi(app=app)`. Errors and access logs will go to `HexServer.txt`.

In [ ]:
#| export
@patch
def html(self:Terrain,wrapper:HexWrapper = None)->str:
    grid = self.hexGrid
    if wrapper is None:
        wrapper = HexWrapper(callBack=HexWrapper.route())
    clearStyle = StyleCSS("HexClear",stroke="none",fill="white",opacity=0.1)
    for i, h in enumerate(grid.hexes):
        grid.hexes[i].style = clearStyle
        #grid.hexes[i].label = str(i)
    #self.colorMap()
    grid.builder.add_style(clearStyle)
    grid.update(wrapper=wrapper,layer_name="hexes")
    for i, l in enumerate(grid.builder.layers):
        logging.info(f"layer[{i}] is {l.name}")
    return grid.builder.xml()

In [ ]:
#| export
@rt
def showMap(session):
    "This is our basic map. It sets up the map and a series of consoles"
    if 'userid' not in session: session['userid'] = random.randint(0, 1_000_000)

    someSVG = "" #globalTree.element(globalPath)

    treeText = P(NotStr(someSVG),id="hexTree")
    terrain = mainBoard.terrain
    grid = terrain.hexGrid
    builder = grid.builder

    terrain.colorMap()
    grid.update()
    terrain.compute_climate()

    builder.layers = []
    terrain.terrainCream()

    
    logging.info("clearing layers")

    builder.adjust("climates", terrain.dottedClimate())
    builder.adjust("settlement",mainBoard.settlements_overlay())
    builder.adjust("countries", mainBoard.countries_overlay())
    builder.adjust("water", mainBoard.world.basins.draw_watersheds())
    builder.adjust("names",mainBoard.names_overlay())
    #builder.adjust("coast",terrain.coastline_svg())

    

    

    mapText = P(NotStr(terrain.html()),id="map")
    return Titled("Hex Map",
        Div(
            # Sidebar
            Div(
                H4("Menu", cls="text-lg font-bold mb-4"),
                treeText
            ),
            # Main area
            Div(
             
                Div(mapText, id="map-area", cls="p-4 overflow-auto flex-1"),
                cls="flex-1 flex flex-col overflow-hidden",height=800
            ),
            cls="flex h-screen"
        )
    )

In [ ]:
#| export
def countryHtml(countryId):
    countries = mainBoard.kingdoms
    showCountry = countries[countryId - 1]
    text = ""
    
    mapDetails  = CountryDetails(showCountry)  
    terrain = mapDetails.countryMap

    grid = terrain.hexGrid
    builder = grid.builder

    grid.adjustRadius(20)

    builder.layers = []

    terrain.colorMap()
    terrain.compute_climate()
    
    terrain.terrainCream()

    

    builder.adjust("climates", terrain.dottedClimate())
    builder.adjust("settlement",mapDetails.settlements_overlay(countries))
    
    builder.adjust("countries",mapDetails.countries_overlay(countries))

    basins = DrainageBasins(terrain)
    #builder.adjust("countries", board.countries_overlay())
    builder.adjust("water", basins.draw_watersheds(max_width=4))
    builder.adjust("names",mapDetails.names_overlay(countries))
    builder.adjust("block",mapDetails.unknownOverlay())
    #wrapper = HexWrapper(callBack=HexWrapper.route())
    detailWrapper = HexWrapper(callBack = lambda grid,index :  {
    "hx-post": f"/country_hex_clicked", #The post gives the route back to the url
    "hx-vals": f'{{"hex_id":{index},"country_id":{showCountry.countryId}}}', # the parameter needs to be called the same in the route
    "hx-target": f"#map" # the element we need to update
    })

    text = NotStr(terrain.html(wrapper=detailWrapper))
    return text


In [ ]:
#| export
@rt
def hex_clicked(session,hex_id: int):
    "This does the raising of the map at a particular point"
    global mainBoard

    terrain = mainBoard.terrain
    grid = terrain.hexGrid
    countries = terrain.fields["country"]

    countryId = int(countries[hex_id])
    if countryId == 0:

        text = f"clicked {hex_id} which is water"
    elif countryId < 0:
        text = f"clicked {hex_id} {countryId} which isn't habitable"
    else:
        text = countryHtml(countryId)
        

    return P(text,id="map")

In [ ]:
@rt
def country_hex_clicked(session,hex_id: int,country_id: int):
    
    text = f"clicked {hex_id} which we need to figure out what country you are looking at"
    countries = mainBoard.kingdoms
    if 0 < country_id <= len(countries):
        showCountry = countries[country_id - 1]
        mapDetails  = CountryDetails(showCountry)
        for country in countries:
            destRegion = mapDetails.viewableRegion(country.region)
            if hex_id in destRegion and country.countryId != country_id:
                return countryHtml( country.countryId )



    text += f" which I am guessing is {showCountry.countryName}."
    return Div(P(text,id="map"), A("back home",href="/showMap"))

In [ ]:
preview(showMap)

In [ ]:
srv.stop()